# Distillation — Colab, one GPU, 100 fixed evaluation cases

This is a separate experiment from the Kaggle notebook. Defaults: Qwen3-8B in 4-bit NF4, **one GPU**, inference batch size 1, **100 saved evaluation cases** across all 18 tasks, and **up to 5 training rounds × 24 new pairs** (up to 120 distinct training pairs). No API calls generate evaluation cases. Training-data generation and audits still use the parent API.

**Persistent storage: Google Drive.** The entire run, API budget, generated training data, predictions, checkpoint state, initial/best adapters and per-round verified ZIPs are written below `MyDrive/HealthInsuranceSLM/colab_adaptive100_unsloth_v1`. Model downloads stay on the temporary Colab disk. GitHub/Kaggle uploads are disabled in this notebook; no GitHub or Kaggle token is needed.

1. In Colab, choose **Runtime → Change runtime type → GPU** (one GPU).
2. Run cells in order. Authorize your Drive mount and upload the accompanying `colab_distillation_bundle.zip` when asked. No GitHub clone/push is required.
3. Add `OPENAI_API_KEY` through Colab Secrets, or use the hidden prompt.
4. Keep the same run name and settings to resume after a disconnect. Do not delete budget/state files.

This is an experimental synthetic benchmark, not expert-reviewed clinical validation. Google Drive authorization and actual CUDA execution must happen in your Colab session; they cannot be validated from this local workspace.


In [ ]:
from pathlib import Path
import os
import sys

if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("Restart the Colab session before running this notebook from the top.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/hf-cache"
for name in ("LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"):
    os.environ.pop(name, None)
# This experiment saves to Drive, independently of the ongoing Kaggle run.
os.environ.pop("KAGGLE_ADAPTER_DATASET", None)
os.environ["GITHUB_ADAPTER_UPLOAD"] = "0"
os.environ["DISTILLATION_DRIVE_BACKUP"] = "1"
os.environ["DISTILLATION_LOCAL_LOCK_DIR"] = "/content/distillation-colab-locks"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/HealthInsuranceSLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive did not mount; do not run training on temporary storage.")
print("Persistent Google Drive folder:", DRIVE_ROOT)


## Upload the companion code bundle

The ZIP contains the current loop code and fixed evaluation corpus, not model weights or credentials. This avoids a stale GitHub checkout. Upload the ZIP supplied with this notebook; it is checked against the expected SHA256 before extraction. Extraction writes only into this experiment's `/content/Health-Insurance-SLM-Colab` code folder, not Google Drive or your Kaggle run.


In [ ]:
import hashlib
import io
import zipfile

EXPECTED_BUNDLE_SHA256 = "fb3638dba104a9fb74674036acfa80726038c8af8775ae8e6e0211f72523e4ca"
BUNDLE = Path("/content/colab_distillation_bundle.zip")
PROJECT_ROOT = Path("/content/Health-Insurance-SLM-Colab")

if not BUNDLE.is_file():
    from google.colab import files
    print("Upload colab_distillation_bundle.zip supplied with this notebook.")
    uploaded = files.upload()
    payload = uploaded.get("colab_distillation_bundle.zip")
    if payload is None:
        raise RuntimeError("Select the supplied colab_distillation_bundle.zip file.")
    BUNDLE.write_bytes(payload)
    del uploaded, payload
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("Bundle and notebook versions differ. Upload the ZIP supplied with this notebook.")
with zipfile.ZipFile(BUNDLE) as bundle:
    for entry in bundle.infolist():
        destination = (Path("/content") / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT_ROOT):
            raise RuntimeError("Unexpected path inside the code bundle.")
        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError("Symbolic links are not allowed in the code bundle.")
    bundle.extractall("/content")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
REQUIREMENTS = str(PROJECT_ROOT / "requirements-colab.txt")
print("Code and fixed evaluation data ready:", PROJECT_ROOT)


In [ ]:
from importlib.metadata import version
import subprocess

# Preserve the CUDA-enabled PyTorch build supplied by Colab.
torch_constraint = Path("/content/distillation-torch-constraint.txt")
torch_constraint.write_text(f"torch=={version('torch')}\n")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", str(torch_constraint),
     "-r", REQUIREMENTS],
    check=True,
)


If Colab asks for a runtime restart after installation, restart it and rerun from the first setup cell. Your Drive run folder is preserved. The uploaded ZIP may need uploading again if the VM was replaced.


In [ ]:
import getpass

if not os.environ.get("OPENAI_API_KEY", "").strip():
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (hidden): ").strip()
if not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("OPENAI_API_KEY is required for teacher training-data generation.")
print("Teacher credential loaded. It will not be written to Google Drive or adapter archives.")


## Small-run settings and data/storage preflight

Defaults are 100 development questions, a 108-case final holdout, and up to 120 training pairs over 5 rounds. Each round adds 24 pairs screened from fresh candidates and replays earlier pairs. The 100 cases cover every task with 5 or 6 questions, preferring distinct reasoning families. The complete 540-case corpus is excluded from training.

The separate API allowance is **$2 estimated USD / 800 attempts**, including repairs and audits. It persists on Drive and can stop generation before a target is reached. Changing limits/settings mid-run requires a new `RUN_NAME`; rerunning does not reset charges. The Kaggle budget and run folder are separate.


In [ ]:
import json
from self_instruct_pipeline.config import load_config
from self_instruct_pipeline.persistence import Store, atomic_json
from distillation_loop.core import load_loop_config
from distillation_loop.colab import colab_configs, save_colab_subset
from distillation_loop.preflight import readiness

RUN_NAME = "colab_adaptive100_unsloth_v1"
RUN_DIR = DRIVE_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
parent_config, loop_config = colab_configs(
    load_config(PROJECT_ROOT / "config/self_instruct.json"),
    load_loop_config(str(PROJECT_ROOT / "config/distillation_loop.json")),
    RUN_DIR,
)
# Validate the profile without overwriting a previous run's configuration.
profile_path = Path("/content/requested_colab_config.json")
atomic_json(profile_path, loop_config)
loop_config = load_loop_config(str(profile_path))
report = readiness(parent_config, loop_config, check_runtime=False)
print("Readiness:", report)
if not report["ready"]:
    raise RuntimeError("Resolve the data/configuration issues above before loading the model.")
schemas = json.loads((PROJECT_ROOT / "data/self_instruct/task_schemas.json").read_text())["tasks"]
subset = save_colab_subset(loop_config, schemas, RUN_DIR)
print("Fixed evaluation cases:", subset["total"], "Per task:", subset["per_task"])

saved_config = RUN_DIR / "colab_config.json"
if saved_config.exists() and json.loads(saved_config.read_text()) != loop_config:
    raise RuntimeError("This Drive run folder has different settings; keep the original settings or choose a new RUN_NAME.")
# Confirm Drive supports the actual journal write and local process-lock path.
probe = Store(RUN_DIR / "storage_probe")
with probe.lock():
    probe.append("probe.jsonl", {"drive_write_test": True})
(probe.directory / "probe.jsonl").unlink()
atomic_json(saved_config, loop_config)
print("Google Drive writes verified:", RUN_DIR)
print("One-GPU training profile:", loop_config["training"])


## Load Qwen3-8B with Unsloth on one GPU

Uses 4-bit NF4 with double quantization, float16 compute, LoRA rank 8 and a 3072-token sequence budget. Unsloth switches between QLoRA training and fast generation for evaluation and diagnostics. Inference batch size is 1.


In [ ]:
# Import Unsloth before Transformers, PEFT or TRL so its patches apply.
import unsloth
import torch
from distillation_loop.colab import UnslothStudentRuntime
from distillation_loop.evaluation import require_training_versions

require_training_versions()
student_runtime = UnslothStudentRuntime.load("Qwen/Qwen3-8B", loop_config)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("Unsloth adapter ready. Inference batch size:", loop_config["inference"]["batch_size"])


## Run or resume — all outputs save to Google Drive

Rerun this cell to resume with the same settings. Initial adapters, completed training checkpoints, generated data, API usage/budget, predictions, per-task scores and loop state are written directly to Drive. Every completed training round also gets a ZIP under `drive_backups/`, verified against per-file SHA256 hashes before recording its receipt. ZIPs include only adapter/tokenizer files, not the base model, credentials or run journals.

A disconnected runtime loses GPU memory and may interrupt the current phase, but completed Drive checkpoints remain available for resume. A currently interrupted optimizer run may have to repeat that unfinished round; completed-round training is reused. Use this run folder from only one Colab runtime at a time. Model writes occur at phase boundaries, not after every optimizer step.

To extend this Unsloth run later, set `TOTAL_ROUNDS` to the desired total in the run cell. Keep the existing Drive folder, API budget and model settings. The controller resumes from the last trained round and retains the best checkpoint across all rounds.


In [ ]:
from distillation_loop.controller import run_loop
from self_instruct_pipeline.budget import APIBudget
from self_instruct_pipeline.usage import export_usage

# Set a larger total to extend this Unsloth run after its initial five rounds.
# None uses the saved limit, or the original profile limit for a fresh run.
TOTAL_ROUNDS = None
if TOTAL_ROUNDS is not None and not (RUN_DIR / "loop_state.json").is_file():
    raise FileNotFoundError(f"Find your existing run state in {RUN_DIR} before extending it.")

budget = APIBudget(RUN_DIR / "api_budget", max_calls=800, max_cost_usd="2.00")
try:
    with budget.activate():
        result = run_loop(student_runtime, parent_config, loop_config, round_limit=TOTAL_ROUNDS)
finally:
    print("Google Drive run folder:", RUN_DIR)
    print("Usage:", export_usage(RUN_DIR))
print("Stop reason:", result["status"])
print("Completed rounds:", result["completed_rounds"])
print("Selected checkpoint:", result.get("selected_checkpoint", result["best_checkpoint"]))
print("Best validation score:", result["best_report"]["macro_task_score"])
print("API budget:", budget.state)


In [ ]:
# Inspect persistent backups and progress without training or API calls.
for receipt_path in sorted((RUN_DIR / "drive_backups").glob("round_*.json")):
    receipt = json.loads(receipt_path.read_text())
    print(receipt["status"], receipt["archive"])
state_path = RUN_DIR / "loop_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    print("Completed rounds:", state["completed_rounds"], "Status:", state["status"])
    for entry in state["history"]:
        print("Round", entry["round"], "macro", entry["scores"]["macro_task_score"])


In [ ]:
# Retry only the latest Drive ZIP export if a backup operation was interrupted.
# This does not retrain the model or regenerate teacher data.
from distillation_loop.drive_backup import save_drive_adapter
completed = sorted(RUN_DIR.glob("round_*/training_complete.json"),
                   key=lambda path: int(path.parent.name.split("_")[-1]))
if completed:
    latest = completed[-1]
    checkpoint = json.loads(latest.read_text())["checkpoint"]
    save_drive_adapter(checkpoint, RUN_DIR, int(latest.parent.name.split("_")[-1]))
else:
    print("No completed round yet. Initial adapter/state may still be available on Drive.")
